# Handwritten Digit & Math Recognition
## Exploratory Data Analysis & Model Evaluation Notebook

Notebook ini mendokumentasikan analisis eksplorasi dataset MNIST, pipeline preprocessing pemusatan (center of mass), arsitektur model Convolutional Neural Network (CNN), serta evaluasi performa model.

In [ ]:
import os
import sys
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import classification_report, confusion_matrix

from src.train import load_mnist_data, DIGIT_CLASSES
from src.preprocess import normalize_to_mnist_format

print(f"TensorFlow Version: {tf.__version__}")

### 1. Memuat Dataset MNIST

In [ ]:
x_train, y_train, x_test, y_test = load_mnist_data()
print(f"Data Latih : {x_train.shape}, Label: {y_train.shape}")
print(f"Data Uji   : {x_test.shape}, Label: {y_test.shape}")

### 2. Distribusi Kelas Angka (0-9)

In [ ]:
unique, counts = np.unique(y_train, return_counts=True)
plt.figure(figsize=(8, 4))
plt.bar(unique, counts, color='#2563EB', edgecolor='#1D4ED8')
plt.xticks(unique, DIGIT_CLASSES)
plt.xlabel('Kelas Angka')
plt.ylabel('Jumlah Sampel')
plt.title('Distribusi Sampel Training MNIST')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

### 3. Visualisasi Contoh Angka Tiap Kelas (0-9)

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 5))
for digit in range(10):
    idx = np.where(y_train == digit)[0][0]
    ax = axes[digit // 5, digit % 5]
    ax.imshow(x_train[idx].squeeze(), cmap='gray')
    ax.set_title(f'Angka: {digit}')
    ax.axis('off')
plt.tight_layout()
plt.show()

### 4. Evaluasi Model CNN Tersimpan (`models/digit_cnn.keras`)

In [ ]:
model = keras.models.load_model('../models/digit_cnn.keras')
model.summary()

test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"\nTest Accuracy: {test_acc * 100:.2f}%")
print(f"Test Loss    : {test_loss:.4f}")

### 5. Confusion Matrix & Classification Report

In [ ]:
y_preds_prob = model.predict(x_test, verbose=0)
y_pred = np.argmax(y_preds_prob, axis=1)

print(classification_report(y_test, y_pred, target_names=DIGIT_CLASSES, digits=4))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 7))
plt.imshow(cm, cmap='Blues')
plt.colorbar()
for i in range(10):
    for j in range(10):
        plt.text(j, i, str(cm[i, j]), ha='center', va='center',
                 color='white' if cm[i, j] > cm.max() / 2 else 'black')
plt.xticks(range(10), DIGIT_CLASSES)
plt.yticks(range(10), DIGIT_CLASSES)
plt.xlabel('Prediksi')
plt.ylabel('Nilai Sebenarnya')
plt.title(f'Confusion Matrix (Akurasi: {test_acc*100:.2f}%)')
plt.show()